# Import Modules

In [2]:
import importlib
import os
import sys

import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns

os.chdir("../")
sys.path.insert(0, os.getcwd())

In [21]:
from morai import models
from morai.dashboard.utils import dashboard_helper as dh
from morai.experience import charters, credibility, eda, experience, tables
from morai.forecast import metrics, preprocessors
from morai.utils import custom_logger, helpers

In [4]:
# update log level if wanting more logging
custom_logger.set_log_level("INFO")

In [5]:
pd.options.display.float_format = "{:,.2f}".format

In [6]:
# default is "plotly_mimetype+notebook", however that takes up space.
# "plotly_mimetype+notebook_connected" seems to save space
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"

# Data

In [7]:
pl_parquet_path = r"files/partition/1821/*/*/*.parquet"

In [8]:
variables = [
    "Observation_Year",
    "Sex",
    "Smoker_Status",
    "Insurance_Plan",
    "Issue_Age",
    "Duration",
    "Face_Amount_Band",
    "Issue_Year",
    "Attained_Age",
    "SOA_Post_Lvl_Ind",
    "Number_of_Pfd_Classes",
    "Preferred_Class",
]
measures = [
    "Amount_Exposed",
    "Policies_Exposed",
    "Death_Claim_Amount",
    "Death_Count",
    "Cen2MomP1wMI_byAmt",
    "Cen2MomP2wMI_byAmt",
]
columns_needed = variables + measures
columns_not_needed = [
    "SOA_Antp_Lvl_TP",
    "SOA_Guar_Lvl_TP",
    "MIB_Flag",
    "Preferred_Indicator",
    "Slct_Ult_Ind",
    "ExpDth_Amt_VBT2015",
    "ExpDth_Amt_VBT2015wMI",
    "ExpDth_Cnt_VBT2015",
    "ExpDth_Cnt_VBT2015wMI",
    "Cen3MomP1wMI_byAmt",
    "Cen3MomP2wMI_byAmt",
    "Cen3MomP3wMI_byAmt",
    "Type_Underwriting_Requirements",
    "Substandard_Indicator",
]

In [42]:
# reading in the dataset
pl.enable_string_cache()
lzdf = pl.scan_parquet(
    pl_parquet_path,
    hive_partitioning=True,
).cast({"Sex": pl.Categorical})

In [43]:
initial_row_count = lzdf.select(pl.len()).collect().item()
print(
    f"row count: {initial_row_count:,} \n"
    f"exposures: {lzdf.select([pl.col('Amount_Exposed').sum()]).collect()[0,0]:,}"
)

row count: 33,552,968 
exposures: 47,112,017,649,521.82


In [44]:
no_values = (
    (pl.col("Amount_Exposed") != 0)
    | (pl.col("Policies_Exposed") != 0)
    | (pl.col("Death_Claim_Amount") != 0)
    | (pl.col("Death_Count") != 0)
)
no_values_row_count = lzdf.filter(no_values).select(pl.len()).collect().item()
print(
    f"removing cells with 'no claim and exposure' which was: {initial_row_count-no_values_row_count:,} cells."
)

removing cells with 'no claim and exposure' which was: 14,658 cells.


In [45]:
claim_with_no_exposure = (pl.col("Death_Claim_Amount") > 0) & (
    pl.col("Amount_Exposed") == 0
)
claim_with_no_exposure_row_count = (
    lzdf.filter(no_values & ~claim_with_no_exposure).select(pl.len()).collect().item()
)
print(
    f"making 'exposure_amount=claim_amount' for cells 'with a claim and no exposure' which was: {no_values_row_count-claim_with_no_exposure_row_count:,} cells."
)

making 'exposure_amount=claim_amount' for cells 'with a claim and no exposure' which was: 72 cells.


In [46]:
no_exposure = pl.col("Amount_Exposed") <= 0
no_exposure_row_count = (
    lzdf.filter(no_values & ~claim_with_no_exposure & ~no_exposure)
    .select(pl.len())
    .collect()
    .item()
)
print(
    f"removing cells with 'no exposure' which was: {claim_with_no_exposure_row_count-no_exposure_row_count:,} cells."
)

removing cells with 'no exposure' which was: 1,085 cells.


In [47]:
lzdf = lzdf.with_columns(
    [
        pl.when(pl.col("Amount_Exposed") < pl.col("Death_Claim_Amount"))
        .then(pl.col("Death_Claim_Amount"))
        .otherwise(pl.col("Amount_Exposed"))
        .alias("Amount_Exposed")
    ]
)
updated_row_count = (
    lzdf.filter(no_values & ~claim_with_no_exposure).select(pl.len()).collect().item()
)
print(f"now there are: {no_values_row_count-updated_row_count:,} cells.")

now there are: 0 cells.


## Filters

### Core

In [48]:
# used in ILEC report
core_filters = (
    (pl.col("Observation_Year") >= 2012)
    & (pl.col("Issue_Age") > 17)
    & (pl.col("SOA_Post_Lvl_Ind") != "PLT")
    & (pl.col("Insurance_Plan") != "Other")
    & (pl.col("Issue_Year") >= 2000)
    & (pl.col("Smoker_Status") != "U")
    & (
        ~pl.col("Face_Amount_Band").is_in(
            (
                "01: 0 - 9,999",
                "02: 10,000 - 24,999",
                "03: 25,000 - 49,999",
                "04: 50,000 - 99,999",
            )
        )
    )
)

### Base

In [49]:
# less strict filter allowing 1980+ and all face bands
base_filters = (
    (pl.col("Observation_Year") >= 2012)
    & (pl.col("Issue_Age") > 17)
    & (pl.col("SOA_Post_Lvl_Ind") != "PLT")
    & (pl.col("Insurance_Plan") != "Other")
    & (pl.col("Issue_Year") >= 1980)
    & (pl.col("Smoker_Status") != "U")
)

## Grouped Datasets

Grouped dataset is grouped on certain variables to limit the amount of memory that is stored when having the entire dataset

### Grouped

In [64]:
# parquet files read 4 - 10x faster
# polars is faster (35 secs)
grouped_df = (
    lzdf.group_by(variables)
    .agg([pl.col(measure).sum() for measure in measures])
    .filter(no_values)
    .filter(~no_exposure)
    .filter(base_filters)
    .collect()
)

In [65]:
grouped_df = grouped_df.to_pandas()

In [66]:
grouped_df = helpers.clean_df(grouped_df)

 2025-08-12 00:45:28 | morai.utils.helpers | INFO     | lowercasing the column names 
 2025-08-12 00:45:28 | morai.utils.helpers | INFO     | replacing special characters with underscores in the column names 
 2025-08-12 00:45:28 | morai.utils.helpers | INFO     | removed unused categories and reorder 
 2025-08-12 00:45:35 | morai.utils.helpers | INFO     | number_of_pfd_classes has missing values, filling with _NULL_ 
 2025-08-12 00:45:37 | morai.utils.helpers | INFO     | preferred_class has missing values, filling with _NULL_ 
 2025-08-12 00:45:38 | morai.utils.helpers | INFO     | update index to int32 
 2025-08-12 00:45:38 | morai.utils.helpers | INFO     | dataFrame shape: (9296716, 18) 


In [67]:
# grouped_df = grouped_df.sample(frac=0.1)

In [68]:
# try to keep below 5M on a 16GB RAM
grouped_df.shape

(9296716, 18)

In [69]:
grouped_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9296716 entries, 0 to 9296715
Data columns (total 18 columns):
 #   Column                 Dtype   
---  ------                 -----   
 0   observation_year       int32   
 1   sex                    category
 2   smoker_status          category
 3   insurance_plan         category
 4   issue_age              int32   
 5   duration               int32   
 6   face_amount_band       category
 7   issue_year             int32   
 8   attained_age           int32   
 9   soa_post_lvl_ind       category
 10  number_of_pfd_classes  category
 11  preferred_class        category
 12  amount_exposed         float64 
 13  policies_exposed       float64 
 14  death_claim_amount     float64 
 15  death_count            int32   
 16  cen2momp1wmi_byamt     float64 
 17  cen2momp2wmi_byamt     float64 
dtypes: category(7), float64(5), int32(6)
memory usage: 665.0 MB


In [70]:
helpers.memory_usage_df(grouped_df)

Total memory usage: 664.9559030532837 mb
Column consuming the most memory: amount_exposed
Memory usage per column:
Index                    37186864
observation_year         37186864
sex                       9296924
smoker_status             9296925
insurance_plan            9297154
issue_age                37186864
duration                 37186864
face_amount_band          9297778
issue_year               37186864
attained_age             37186864
soa_post_lvl_ind          9297096
number_of_pfd_classes     9297093
preferred_class           9297143
amount_exposed           74373728
policies_exposed         74373728
death_claim_amount       74373728
death_count              37186864
cen2momp1wmi_byamt       74373728
cen2momp2wmi_byamt       74373728
dtype: int64


In [71]:
nan_counts = grouped_df.isna().sum()
nan_counts

observation_year         0
sex                      0
smoker_status            0
insurance_plan           0
issue_age                0
duration                 0
face_amount_band         0
issue_year               0
attained_age             0
soa_post_lvl_ind         0
number_of_pfd_classes    0
preferred_class          0
amount_exposed           0
policies_exposed         0
death_claim_amount       0
death_count              0
cen2momp1wmi_byamt       0
cen2momp2wmi_byamt       0
dtype: int64

## Enhance Dataset

### Group Fields

In [72]:
grouped_df["class_enh"] = (
    grouped_df["number_of_pfd_classes"].astype(str)
    + "_"
    + grouped_df["preferred_class"].astype(str)
).astype("category")

### Add Mortality Table

In [73]:
grouped_df = tables.map_rates(
    df=grouped_df,
    rate="vbt15",
)

 2025-08-12 00:45:54 | morai.experience.tables | INFO     | mapping rate: 'qx_vbt15' with format: 'soa' 
 2025-08-12 00:46:01 | morai.experience.tables | INFO     | the mapped rates are based on the following keys: ['issue_age', 'duration', 'sex', 'smoker_status'] 


### Calculated Fields

In [74]:
grouped_df["qx_raw"] = np.where(
    grouped_df["amount_exposed"] == 0,
    0,
    grouped_df["death_claim_amount"] / grouped_df["amount_exposed"],
)

In [75]:
print(
    f"There were {len(grouped_df[grouped_df['qx_raw']>1])} records that were greater than 1. This should not happen with annual exposure method. Capping at 1"
)
print(
    f"There were {len(grouped_df[grouped_df['amount_exposed']==0])} records that had 0 exposure with a claim or a policy count. This shouldn't happen"
)
print(
    f"There were {len(grouped_df[(grouped_df['amount_exposed']==0)&(grouped_df['death_claim_amount']>0)])} records that had 0 exposure with a claim. This shouldn't happen"
)
grouped_df["qx_raw"] = grouped_df["qx_raw"].clip(upper=1)
grouped_df["qx_log_raw"] = np.log(grouped_df["qx_raw"] + 1)
grouped_df["exp_amt_vbt15"] = grouped_df["qx_vbt15"] * grouped_df["amount_exposed"]
grouped_df["ae_vbt15"] = np.where(
    grouped_df["amount_exposed"] == 0,
    0,
    grouped_df["death_claim_amount"] / grouped_df["exp_amt_vbt15"],
)

There were 0 records that were greater than 1. This should not happen with annual exposure method. Capping at 1
There were 0 records that had 0 exposure with a claim or a policy count. This shouldn't happen
There were 0 records that had 0 exposure with a claim. This shouldn't happen


In [76]:
for col in grouped_df.select_dtypes(include="object").columns:
    grouped_df[col] = grouped_df[col].astype("category")

In [77]:
# write to file for predictive model
grouped_df.to_parquet("files/dataset/full_mortality_grouped_1821.parquet")

## Load Data

In [9]:
pl_parquet_path = r"files/dataset/full_mortality_grouped_1821.parquet"

In [10]:
# reading in the dataset
# `enable_string_cache` helps with categorical type values
pl.enable_string_cache()
lzdf = pl.scan_parquet(
    pl_parquet_path,
)

In [11]:
initial_row_count = lzdf.select(pl.len()).collect().item()
print(
    f"row count: {initial_row_count:,} \n"
    f"exposures: {lzdf.select([pl.col('amount_exposed').sum()]).collect()[0,0]:,}"
)

row count: 9,296,716 
exposures: 44,538,135,193,210.53


In [12]:
lzdf.select([pl.col("amount_exposed").sum()]).collect()[0, 0]

44538135193210.53

In [13]:
grouped_df = lzdf.collect()

In [14]:
grouped_df = grouped_df.to_pandas()

# Exploratory

In [37]:
variables = [variable.lower() for variable in variables + ["class_enh"]]
measures = [measure.lower() for measure in measures]

## Descriptive

### Numeric

In [38]:
desc = grouped_df.describe()
desc.loc["sum"] = grouped_df.select_dtypes(include=[np.number]).sum()
desc

,observation_year,issue_age,duration,issue_year,attained_age,amount_exposed,policies_exposed,death_claim_amount,death_count,cen2momp1wmi_byamt,cen2momp2wmi_byamt,qx_vbt15,qx_raw,qx_log_raw,exp_amt_vbt15,ae_vbt15
count,"17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00","17,934,144.00"
mean,"2,015.85",44.90,12.57,"2,003.79",56.47,"5,510,949.57",16.99,"13,749.38",0.09,"38,119,195,226.72","2,270,198,214.64",0.01,0.01,0.01,"15,309.91",1.00
std,2.31,15.59,8.35,8.40,16.77,"28,082,486.50",56.31,"193,542.47",0.63,"1,091,536,980,450.54","124,075,240,070.47",0.03,0.07,0.06,"72,687.44",22.36
min,"2,012.00",18.00,1.00,"1,981.00",18.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,"2,014.00",32.00,6.00,"1,999.00",44.00,"92,458.06",0.74,0.00,0.00,"18,967,204.00","34,133.54",0.00,0.00,0.00,243.09,0.00
50%,"2,016.00",44.00,11.00,"2,005.00",56.00,"491,256.81",2.09,0.00,0.00,"261,500,824.00","694,462.59",0.00,0.00,0.00,"1,421.56",0.00
75%,"2,018.00",56.00,18.00,"2,010.00",69.00,"2,389,041.00",8.43,0.00,0.00,"2,832,966,736.00","11,851,092.59",0.01,0.00,0.00,"7,267.31",0.00
max,"2,019.00",100.00,39.00,"2,019.00",120.00,"2,023,852,754.17","1,611.29","75,000,000.00",44.00,"768,935,041,433,600.00","113,476,559,175,680.00",0.50,1.00,0.69,"11,007,973.30","14,285.71"
sum,"36,152,482,443.00","805,191,379.00","225,421,795.00","35,936,195,877.00","1,012,679,030.00","98,834,163,081,499.58","304,679,453.49","246,583,325,845.00","1,688,627.00","683,635,136,360,133,376.00","40,714,061,689,889,728.00","220,162.71","175,357.60","141,056.10","274,570,160,219.11","17,876,628.93"


In [39]:
charters.frequency(
    grouped_df,
    cols=3,
    features=grouped_df[variables].select_dtypes(include=["number"]).columns.tolist(),
    sum_var="amount_exposed",
).show()

### Categorical

In [40]:
charters.frequency(grouped_df, cols=3, sum_var="policies_exposed").show()

 2025-05-12 20:10:02 | morai.experience.charters | INFO     | No features provided, using all non-numeric features. 


### Target

In [41]:
charters.target(
    df=grouped_df,
    target="risk",
    cols=3,
    features=variables,
    numerator=["death_claim_amount"],
    denominator=["amount_exposed"],
).show()

 2025-05-12 20:10:14 | morai.experience.charters | INFO     | Creating '14' target plots. 


In [42]:
charters.target(
    df=grouped_df,
    target="risk",
    cols=3,
    features=variables,
    numerator=["death_claim_amount"],
    denominator=["amount_exposed"],
).show()

 2025-05-12 20:10:28 | morai.experience.charters | INFO     | Creating '78' target plots. 


### Other

Correlation provides the relationship that two features have together. A correlation of 1 indicates they move in the same direction, and a correlation of 0 indicates they do not have a relationship.

In [43]:
corr_num = eda.correlation(
    df=grouped_df, features=variables + ["qx_raw"], method="kendall", numeric=True
)
charters.matrix(df=corr_num, threshold=0.5, title="Numeric Correlation Matrix")

 2025-05-12 20:15:10 | morai.experience.eda | INFO     | Creating correlation matrix for numeric features using method: 'kendall'. 


In [44]:
corr_cat = eda.correlation(
    df=grouped_df, features=variables + ["qx_raw"], method="kendall", numeric=False
)
charters.matrix(df=corr_cat, threshold=0.5, title="Categorical Correlation Matrix")

 2025-05-12 20:17:58 | morai.experience.eda | INFO     | Creating correlation matrix for categorical features using cramers_v.  


Mutual information is similar to correlation and determines the strength of two variables relationship

In [ ]:
mi = eda.mutual_info(
    df=grouped_df,
    features=variables + ["qx_raw"],
    n_jobs=-1,
)
charters.matrix(df=mi, threshold=0.3, title="Mutual Information Matrix")

GVIF determines the amount of multicollinearity the features have. A value of 5 or 10 are standard thresholds.

In [46]:
# these features cause gvif to be too high
# "number_of_pfd_classes"
# "preferred_class"
# "issue_year"
# "issue_age"

# these features aren't needed
# "soa_post_lvl_ind"

gvif = eda.gvif(
    df=grouped_df,
    features=[
        "attained_age",
        "duration",
        "observation_year",
        "sex",
        "smoker_status",
        "face_amount_band",
        "insurance_plan",
        "class_enh",
    ],
    numeric_only=False,
)
gvif

 2025-05-12 21:53:31 | morai.experience.eda | INFO     | Converting '5' categorical features to dummy variables. 
 2025-05-12 21:53:36 | morai.experience.eda | INFO     | Calculating the GVIF for '8' features. 


,feature,gvif
0,attained_age,1.23
1,duration,1.42
2,observation_year,1.07
3,sex,1.00
4,smoker_status,1.13
5,face_amount_band,1.01
6,insurance_plan,1.01
7,class_enh,1.03


## Mortality Explore

Mortality qx seems to be increasing by observation year, however as we shall see this is just a function of age

In [47]:
charters.chart(
    df=grouped_df,
    x_axis="observation_year",
    y_axis="ratio",
    color=None,
    type="line",
    numerator="death_claim_amount",
    denominator="amount_exposed",
)

 2025-05-12 22:09:54 | morai.experience.charters | INFO     | Calculating ratio using [death_claim_amount] and [amount_exposed] 


In [48]:
charters.chart(
    df=grouped_df,
    x_axis="observation_year",
    y_axis="risk",
    color="sex",
    type="line",
    numerator="death_claim_amount",
    denominator="amount_exposed",
)

 2025-05-12 22:09:58 | morai.experience.charters | INFO     | Calculating risk using [death_claim_amount] and [amount_exposed] 


In [49]:
charters.chart(
    df=grouped_df,
    x_axis="attained_age",
    y_axis="qx_log_raw",
    type="line",
    agg="mean",
)

In [50]:
charters.chart(
    df=grouped_df,
    x_axis="attained_age",
    y_axis="duration",
    color="death_count",
    type="heatmap",
)

In [51]:
charters.compare_rates(
    grouped_df[grouped_df["insurance_plan"].isin(["UL", "ULSG"])],
    x_axis="attained_age",
    rates=["ae_vbt15"],
    weights=["exp_amt_vbt15"],
    secondary="death_count",
    y_log=False,
    x_bins=11,
    display=True,
)

 2025-05-12 22:10:11 | morai.experience.charters | INFO     | Binning feature: [attained_age] with 11 bins 


In [52]:
charters.compare_rates(
    grouped_df[grouped_df["insurance_plan"].isin(["UL", "ULSG"])],
    x_axis="duration",
    rates=["ae_vbt15"],
    weights=["exp_amt_vbt15"],
    secondary="death_count",
    y_log=False,
    x_bins=11,
    display=True,
)

 2025-05-12 22:10:16 | morai.experience.charters | INFO     | Binning feature: [duration] with 11 bins 


# Reload

In [110]:
importlib.reload(charters)

<module 'morai.experience.charters' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\experience\\charters.py'>

In [ ]:
importlib.reload(custom_logger)

In [ ]:
importlib.reload(forecasters)

In [ ]:
importlib.reload(helpers)

In [ ]:
importlib.reload(metrics)

In [87]:
importlib.reload(experience)

<module 'morai.experience.experience' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\experience\\experience.py'>

# Utilities

In [55]:
del grouped_df

In [56]:
helpers.memory_usage_jupyter().head(10)

,object,size_mb
0,corr_cat,0.00
1,nan_counts,0.00
2,desc,0.00
3,corr_num,0.00
4,gvif,0.00
5,columns_needed,0.00
6,variables,0.00
7,open,0.00
8,columns_not_needed,0.00
9,measures,0.00


In [61]:
helpers.memory_usage_jupyter_cells("02.exploratory.ipynb")

 2025-05-12 22:16:10 | morai.utils.helpers | INFO     | getting largest cells for C:\Users\johnk\Desktop\github\morai\notebooks\02.exploratory.ipynb 


,cell_idx,cell_type,size_kb,preview
58,58,code,"1,614.70","charters.target( df=grouped_df, target..."
72,72,code,105.30,"charters.chart( df=grouped_df, x_axis=..."
57,57,code,43.00,"charters.target( df=grouped_df, target..."
53,53,code,33.30,"charters.frequency( grouped_df, cols=3..."
55,55,code,30.50,"charters.frequency(grouped_df, cols=3, sum_var..."
62,62,code,26.00,"corr_cat = eda.correlation( df=grouped_df,..."
61,61,code,22.70,"corr_num = eda.correlation( df=grouped_df,..."
71,71,code,21.90,"charters.chart( df=grouped_df, x_axis=..."
70,70,code,20.00,"charters.chart( df=grouped_df, x_axis=..."
73,73,code,19.40,charters.compare_rates( grouped_df[grouped...


# Test